# Gemma 4 — Visual Novel Translation Finetuning

Two-stage LoRA finetuning for Japanese-to-English visual novel localization.

- **Stage 1**: Broad JA/EN instruction behavior (Shisa v2)
- **Stage 2**: VN specialization (VNTL-v3.1-1k-q) with mandatory Shisa replay

Supports model variants: `E4B`, `27B`, `26B-A4B`. Change `MODEL_VARIANT` below.


In [ ]:
%%capture
# Install/upgrade Unsloth and common training dependencies
%pip install -U pip setuptools wheel

# Pin CUDA-enabled PyTorch to a version compatible with current Unsloth
%pip install --upgrade --force-reinstall --no-cache-dir --index-url https://download.pytorch.org/whl/cu128 torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0

%pip install -U unsloth trl datasets accelerate peft bitsandbytes sentencepiece protobuf
%pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

# Optional but commonly useful with Unsloth on NVIDIA GPUs
%pip install -U xformers

print("Dependencies installed. Restart the kernel, then continue.")


## CPU-Only Pre-Download

Run this cell on a **CPU runtime** to download models and datasets before switching to GPU.
This saves GPU billing time on Colab.


In [ ]:
# --- CPU-Only Pre-Download (run on CPU runtime, then switch to GPU) ---

import os

IS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
CACHE_DIR = "/content/model_cache" if IS_COLAB else None

# ── HuggingFace Token (enables faster gated downloads) ───────────────────────
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxxxxxxxxxx"

if HF_TOKEN is None:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None and IS_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"HF_TOKEN set ({HF_TOKEN[:8]}...)")
else:
    print("No HF_TOKEN found. Downloads may be slower for gated models.")

# ── Model Selection ──────────────────────────────────────────────────────────
MODEL_VARIANT = "31B"  # Options: "E4B", "31B", "26B-A4B"

# VRAM estimates (4-bit QLoRA unless noted):
#   E4B:      ~10 GB @ 8k ctx, ~22 GB @ 32k, ~46 GB @ 80k  (L4/A100 friendly)
#   31B:      ~28 GB @ 4k ctx, ~36 GB @ 8k, ~52 GB @ 16k   (A100 40/80GB only)
#   26B-A4B:  ~58 GB @ 2k ctx (bf16 required, MoE incompatible with 4-bit)
#
# 80k context note: Gemma 4 natively supports 128k-256k context. LoRA preserves
# long-context ability even when training on shorter samples (2-8k). Setting
# max_seq_length to your data's p99 + headroom is optimal. Only increase if your
# training samples are genuinely long.
_MODEL_REGISTRY = {
    #              (model_id,                          max_seq, lora_r, lora_alpha, s1_lr,  s2_lr, s1_steps, s2_steps, grad_accum, load_4bit)
    "E4B":        ("unsloth/gemma-4-E4B-it",           4096,    16,     32,        1.5e-4, 5e-5,  600,      800,      4,          True),
    "31B":        ("unsloth/gemma-4-31B-it",            8192,    16,     32,        5e-5,   2e-5,  300,      400,      8,          True),
    "26B-A4B":   ("unsloth/gemma-4-26B-A4B-it",        4096,    16,     32,        1e-4,   4e-5,  300,      400,      8,          False),
}

MODEL_NAME = _MODEL_REGISTRY[MODEL_VARIANT][0]

from huggingface_hub import snapshot_download
from datasets import load_dataset

download_kwargs = {"cache_dir": CACHE_DIR} if CACHE_DIR else {}

print(f"Downloading model: {MODEL_NAME}")
snapshot_download(MODEL_NAME, ignore_patterns=["*.md", "*.txt"], token=HF_TOKEN, **download_kwargs)

print("Downloading datasets...")
load_dataset("shisa-ai/shisa-v2-sharegpt", split="train[:10]")

print("\nAll downloads complete. Switch to GPU runtime now.")


HF_TOKEN set (hf_ImyYD...)


c:\Users\li\miniconda3\envs\unsloth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 10 files: 100%|██████████| 10/10 [00:00<?, ?it/s]


All downloads complete. Switch to GPU runtime now.


## Config


In [2]:
import os, sys

IS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

# ── Model Selection ──────────────────────────────────────────────────────────
MODEL_VARIANT = "31B"  # Options: "E4B", "31B", "26B-A4B"

# VRAM estimates (4-bit QLoRA unless noted):
#   E4B:      ~10 GB @ 8k ctx, ~22 GB @ 32k, ~46 GB @ 80k  (L4/A100 friendly)
#   31B:      ~28 GB @ 4k ctx, ~36 GB @ 8k, ~52 GB @ 16k   (A100 40/80GB only)
#   26B-A4B:  ~58 GB @ 2k ctx (bf16 required, MoE incompatible with 4-bit)
#
# 80k context note: Gemma 4 natively supports 128k-256k context. LoRA preserves
# long-context ability even when training on shorter samples (2-8k). Setting
# max_seq_length to your data's p99 + headroom is optimal. Only increase if your
# training samples are genuinely long.
_MODEL_REGISTRY = {
    #              (model_id,                          max_seq, lora_r, lora_alpha, s1_lr,  s2_lr, s1_steps, s2_steps, grad_accum, load_4bit)
    "E4B":        ("unsloth/gemma-4-E4B-it",           8192,    16,     32,        1.5e-4, 5e-5,  600,      800,      4,          True),
    "31B":        ("unsloth/gemma-4-31B-it",            8192,    16,     32,        5e-5,   2e-5,  300,      400,      8,          True),
    "26B-A4B":   ("unsloth/gemma-4-26B-A4B-it",        4096,    16,     32,        1e-4,   4e-5,  300,      400,      8,          False),
}

(MODEL_NAME, MAX_SEQ_LENGTH, LORA_R, LORA_ALPHA,
 STAGE1_LR, STAGE2_LR, STAGE1_MAX_STEPS, STAGE2_MAX_STEPS,
 GRAD_ACCUM_STEPS, LOAD_IN_4BIT) = _MODEL_REGISTRY[MODEL_VARIANT]

RANDOM_SEED = 3407

# ── Datasets ─────────────────────────────────────────────────────────────────
SHISA_DATASET = "shisa-ai/shisa-v2-sharegpt"
VNTL_LOCAL_JSONL = "assets/lmg-anon__VNTL-v3.1-1k.jsonl"

STAGE1_SHISA_ROWS = 12000
SHISA_REPLAY_ROWS = 3000
SHISA_HOLDOUT_ROWS = 256
SHISA_HOLDOUT_POOL_ROWS = 1024
VNTL_Q_DEV_ROWS = 256
VNTL_Q_STRESS_ROWS = 128
HARUBENCH_EVAL_ROWS = 256

# ── Checkpoints & Retention ──────────────────────────────────────────────────
CHECKPOINT_ORDER = ("base", "stage1_shisa", "stage2_vntl_replay")
RETENTION_TRACK_PREFIX = "retention_shisa"
RETENTION_EVAL_LOSS_TOLERANCE = 0.20
RETENTION_PERPLEXITY_TOLERANCE = 5.0

# ── NSFW Filtering ───────────────────────────────────────────────────────────
ENABLE_NSFW_FILTER = True
APPLY_NSFW_FILTER_TO_STAGE2 = True
APPLY_NSFW_FILTER_TO_EVAL_TRACKS = True
NSFW_FILTER_LEVEL_STAGE2 = "explicit_only"
NSFW_FILTER_LEVEL_EVAL_TRACKS = "minimal"

# ── Multi-GPU ────────────────────────────────────────────────────────────────
USE_MULTI_GPU_MODEL_SHARDING = True
MULTI_GPU_DEVICE_MAP = "balanced"
GPU0_RESERVED_GIB = 5
OTHER_GPU_RESERVED_GIB = 3

# ── Directories ──────────────────────────────────────────────────────────────
OUTPUT_DIR = "outputs/stage1"
STAGE2_OUTPUT_DIR = "outputs/stage2"
FINAL_ADAPTER_DIR = f"outputs/gemma4_{MODEL_VARIANT.lower()}_lora_final"
EVAL_EXPORT_DIR = "eval_exports"

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        FINAL_ADAPTER_DIR = f"/content/drive/MyDrive/vn_finetune/{MODEL_VARIANT}_lora"
        print(f"Colab detected. Final adapter will save to: {FINAL_ADAPTER_DIR}")
    except Exception as e:
        print(f"Drive mount skipped: {e}")

print(f"Model: {MODEL_NAME} | Variant: {MODEL_VARIANT}")
print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} | Seq len={MAX_SEQ_LENGTH}")
print(f"Stage1: {STAGE1_MAX_STEPS} steps @ lr={STAGE1_LR} | Stage2: {STAGE2_MAX_STEPS} steps @ lr={STAGE2_LR}")


Model: unsloth/gemma-4-31B-it | Variant: 31B
LoRA r=16 alpha=32 | Seq len=8192
Stage1: 300 steps @ lr=5e-05 | Stage2: 400 steps @ lr=2e-05


## GPU Setup


In [3]:
import os, subprocess

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

available_gpus = []
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"],
        text=True,
    )
    available_gpus = [line.split(",", 1)[0].strip() for line in smi_output.splitlines() if line.strip()]
except Exception as e:
    print("nvidia-smi query failed:", e)

if available_gpus:
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(available_gpus)
    print("Using all available GPUs:", os.environ["CUDA_VISIBLE_DEVICES"])
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    print("No GPUs were reported by nvidia-smi.")

import torch

print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    x = torch.randn(2048, device=f"cuda:{i}")
    y = (x * x).mean().item()
    print(f"  cuda:{i} {props.name} | VRAM {props.total_memory / 1024**3:.2f} GiB | compute check ok (mean={y:.4f})")
    del x


Using all available GPUs: 0,1
CUDA available: True
Visible GPU count: 2
  cuda:0 NVIDIA GeForce RTX 5090 | VRAM 31.84 GiB | compute check ok (mean=0.9926)
  cuda:1 NVIDIA GeForce RTX 3090 | VRAM 24.00 GiB | compute check ok (mean=1.0108)


## Imports

In [4]:
import sys
sys.path.insert(0, "src")

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import standardize_data_formats

from vn_finetune_utils import (
    build_eval_tracks,
    export_jsonl_rows,
    has_placeholders,
    print_checkpoint_summary,
    shisa_stratify_key,
    stratified_select,
    summarize_checkpoint_metrics,
    vntl_stratify_key,
)
from vn_nsfw import (
    is_nsfw_row as _is_nsfw_row_impl,
    is_nsfw_text as _is_nsfw_text_impl,
    load_keyword_scores,
    nsfw_score as _nsfw_score_impl,
)
from vn_eval_suite import run_targeted_checks, summarize_targeted_results

try:
    from trl import SFTTrainer, SFTConfig
except Exception:
    from trl.trainer.sft_trainer import SFTTrainer
    from trl.trainer.sft_config import SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0404 04:25:28.930000 79360 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Dataset Loading

In [5]:
from pathlib import Path
from collections import OrderedDict


def _load_dataset_split(repo_ids, split):
    candidate_ids = [repo_ids] if isinstance(repo_ids, str) else list(repo_ids)
    last_error = None
    for repo_id in candidate_ids:
        try:
            ds = load_dataset(repo_id, split=split, token=HF_TOKEN)
            return repo_id, ds
        except Exception as exc:
            last_error = exc
            print(f"Failed to load {repo_id} [{split}]: {exc}")
    raise RuntimeError(f"Unable to load split {split} from any candidate repo: {candidate_ids}") from last_error


def _load_jsonl_dataset(jsonl_path):
    resolved = Path(jsonl_path).expanduser().resolve()
    ds = load_dataset("json", data_files=str(resolved), split="train")
    return str(resolved), ds


def _standardize_dataset(ds):
    return standardize_data_formats(ds)


_, shisa_stage1_raw = _load_dataset_split(SHISA_DATASET, f"train[:{STAGE1_SHISA_ROWS}]")
_, shisa_replay_raw = _load_dataset_split(SHISA_DATASET, f"train[{STAGE1_SHISA_ROWS}:{STAGE1_SHISA_ROWS + SHISA_REPLAY_ROWS}]")
_, shisa_holdout_pool_raw = _load_dataset_split(SHISA_DATASET, f"train[{STAGE1_SHISA_ROWS + SHISA_REPLAY_ROWS}:{STAGE1_SHISA_ROWS + SHISA_REPLAY_ROWS + SHISA_HOLDOUT_POOL_ROWS}]")

# Load all VNTL data from local JSONL and split into train/val/stress/harubench
_, vntl_all_raw = _load_jsonl_dataset(VNTL_LOCAL_JSONL)
vntl_all_raw = vntl_all_raw.shuffle(seed=RANDOM_SEED)
_vntl_total = len(vntl_all_raw)

# Split: last VNTL_Q_STRESS_ROWS -> stress, next VNTL_Q_DEV_ROWS -> val,
#         next HARUBENCH_EVAL_ROWS -> harubench eval, rest -> train
_stress_start = _vntl_total - VNTL_Q_STRESS_ROWS
_val_start = _stress_start - VNTL_Q_DEV_ROWS
_harubench_start = _val_start - HARUBENCH_EVAL_ROWS

vntl_q_train_raw = vntl_all_raw.select(range(0, _harubench_start))
vntl_harubench_raw = vntl_all_raw.select(range(_harubench_start, _val_start))
vntl_q_val_raw = vntl_all_raw.select(range(_val_start, _stress_start))
vntl_q_stress_raw = vntl_all_raw.select(range(_stress_start, _vntl_total))

print(f"VNTL local: {_vntl_total} total -> train={len(vntl_q_train_raw)} harubench={len(vntl_harubench_raw)} val={len(vntl_q_val_raw)} stress={len(vntl_q_stress_raw)}")

shisa_stage1_raw = _standardize_dataset(shisa_stage1_raw)
shisa_replay_raw = _standardize_dataset(shisa_replay_raw)
shisa_holdout_pool_raw = _standardize_dataset(shisa_holdout_pool_raw)
vntl_q_train_raw = _standardize_dataset(vntl_q_train_raw)
vntl_q_stress_raw = _standardize_dataset(vntl_q_stress_raw)
vntl_q_val_raw = _standardize_dataset(vntl_q_val_raw)
vntl_harubench_raw = _standardize_dataset(vntl_harubench_raw)
print(f"VNTL source: {VNTL_LOCAL_JSONL}")

stage2_raw = concatenate_datasets([vntl_q_train_raw, shisa_replay_raw]).shuffle(seed=RANDOM_SEED)

RAW_EVAL_POOLS = OrderedDict([
    ("retention_shisa", shisa_holdout_pool_raw),
    ("vntl_q_dev", vntl_q_val_raw),
    ("vntl_q_stress", vntl_q_stress_raw),
    ("vntl_harubench", vntl_harubench_raw),
])

print(f"Stage1 Shisa rows: {len(shisa_stage1_raw)} | Stage2 rows: {len(stage2_raw)} (VNTL-q train + replay)")
print(f"Replay rows: {len(shisa_replay_raw)} | Shisa holdout pool: {len(shisa_holdout_pool_raw)}")
print(f"VNTL val: {len(vntl_q_val_raw)} | VNTL stress: {len(vntl_q_stress_raw)} | VNTL harubench: {len(vntl_harubench_raw)}")


VNTL local: 4035 total -> train=3395 harubench=256 val=256 stress=128
VNTL source: assets/lmg-anon__VNTL-v3.1-1k.jsonl
Stage1 Shisa rows: 12000 | Stage2 rows: 6395 (VNTL-q train + replay)
Replay rows: 3000 | Shisa holdout pool: 1024
VNTL val: 256 | VNTL stress: 128 | VNTL harubench: 256


## Prompt Formatting

System prompt mirrors the production translation pipeline to ensure training/inference alignment.
VNTL data is structured with `<character_reference>` tags matching the production format.


In [ ]:
from transformers import AutoTokenizer
import re

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
    print("Tokenizer initialized for Gemma-4 chat formatting.")


# The Gemma4Processor cannot apply_chat_template with system role content.
# Use the inner tokenizer (GemmaTokenizerFast) for chat template rendering.
_inner_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

NSFW_KEYWORD_SCORES = load_keyword_scores()
print(f"Loaded {len(NSFW_KEYWORD_SCORES)} NSFW keyword entries")

# ── System prompt aligned with production pipeline ───────────────────────────
VN_TRAINING_SYSTEM_PROMPT = """You are an expert Japanese-to-English visual novel localization translator.

Translate the Japanese script lines into natural, publication-quality English that reads as if originally written in English.

Translation principles:
- Convey the speaker's intent, emotion, and subtext \u2014 not just literal meaning
- Adapt Japanese speech patterns to natural English equivalents:
  \u00b7 Formal/polite speech (\u3067\u3059/\u307e\u3059) \u2192 appropriate English register
  \u00b7 Rough/masculine speech (\u4ffa, \u3060\u305c, etc.) \u2192 casual/rough English
  \u00b7 Feminine/soft speech (\u308f, \u306e, etc.) \u2192 natural feminine English without stereotype
  \u00b7 Child speech patterns \u2192 age-appropriate English
- Preserve humor, wordplay, and cultural references where possible; adapt rather than transliterate
- Keep honorifics when they serve characterization; otherwise adapt naturally
- Maintain continuity with scene context and character relationships
- Keep engine tags, placeholders, and formatting exactly as-is
- Output only the English localization \u2014 no commentary, no Japanese text"""


def _is_non_empty_text(x):
    return isinstance(x, str) and len(x.strip()) > 0


def _get_row_value(batch, key, idx):
    values = batch.get(key, None)
    if isinstance(values, list) and idx < len(values):
        return values[idx]
    return None


def _clean_vntl_text(text):
    if not isinstance(text, str):
        return ""
    cleaned = text
    cleaned = cleaned.replace("%name", "<NAME>")
    cleaned = cleaned.replace("%nick", "<NICK>")
    cleaned = cleaned.replace("%fp", "<FP>")
    cleaned = re.sub(r"<\|[^>]+\|>", "", cleaned)
    return cleaned.strip()


def _ignore_segments(ignore_loss):
    if not isinstance(ignore_loss, list):
        return set()
    segs = set()
    for value in ignore_loss:
        if isinstance(value, (int, float)) and int(value) >= 0:
            segs.add(int(value))
    return segs


def _is_nsfw_text(text):
    return _is_nsfw_text_impl(text, level="moderate", keyword_scores=NSFW_KEYWORD_SCORES)


def _is_nsfw_row(row):
    return _is_nsfw_row_impl(row, level="moderate", keyword_scores=NSFW_KEYWORD_SCORES)


def _is_nsfw_text_tuned(text, level="explicit_only"):
    return _is_nsfw_text_impl(text, level=level, keyword_scores=NSFW_KEYWORD_SCORES)


def _is_nsfw_row_tuned(row, level="explicit_only"):
    return _is_nsfw_row_impl(row, level=level, keyword_scores=NSFW_KEYWORD_SCORES)


def _extract_vntl_pair(text, ignore_loss=None):
    """Extract metadata + JP/EN pair from VNTL format, returning (metadata, ja_text, en_text)."""
    if not isinstance(text, str):
        return None
    if "<<JAPANESE>>" not in text or "<<ENGLISH>>" not in text:
        return None

    # Extract metadata block
    metadata = ""
    meta_match = re.search(r"<<METADATA>>\n(.*?)\n<<START>>", text, flags=re.DOTALL)
    if meta_match:
        metadata = meta_match.group(1).strip()

    pair_matches = list(re.finditer(
        r"<<JAPANESE>>(.*?)<<ENGLISH>>(.*?)(?=<<JAPANESE>>|$)",
        text, flags=re.DOTALL,
    ))
    if not pair_matches:
        return None

    ignored_segment_ids = _ignore_segments(ignore_loss)
    ja_segments = []
    en_segments = []
    for seg_idx, match in enumerate(pair_matches):
        if seg_idx in ignored_segment_ids:
            continue
        ja_text = _clean_vntl_text(match.group(1))
        en_text = _clean_vntl_text(match.group(2))
        if _is_non_empty_text(ja_text) and _is_non_empty_text(en_text):
            ja_segments.append(ja_text)
            en_segments.append(en_text)

    if not ja_segments or not en_segments:
        return None

    return metadata, "\n\n".join(ja_segments), "\n\n".join(en_segments)


def _render_vntl_chat(metadata, ja_text, en_text):
    """Render VNTL pair as chat with system prompt + character context."""
    user_content = ""
    if metadata:
        user_content += f"<character_reference>\n{metadata}\n</character_reference>\n\n"
    user_content += f"Japanese script:\n{ja_text}"

    convo = [
        {"role": "system", "content": VN_TRAINING_SYSTEM_PROMPT},
        {"role": "user", "content": user_content.strip()},
        {"role": "assistant", "content": en_text.strip()},
    ]
    try:
        return _inner_tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False,
        ).removeprefix('<bos>')
    except Exception:
        return f"{user_content.strip()}\n{en_text.strip()}"


def _render_chat_from_pair(user_text, assistant_text):
    """Render a generic user/assistant pair as chat text."""
    convo = [
        {"role": "user", "content": user_text.strip()},
        {"role": "assistant", "content": assistant_text.strip()},
    ]
    try:
        return _inner_tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False,
        ).removeprefix('<bos>')
    except Exception:
        return f"{user_text.strip()}\n{assistant_text.strip()}"


def _row_to_text_from_batch(batch, idx):
    # Conversations (Shisa format)
    convo = _get_row_value(batch, "conversations", idx)
    if isinstance(convo, list) and len(convo) > 0:
        try:
            return _inner_tokenizer.apply_chat_template(
                convo, tokenize=False, add_generation_prompt=False,
            ).removeprefix('<bos>')
        except Exception:
            pass

    # VNTL format with system prompt
    raw_text = _get_row_value(batch, "text", idx)
    raw_ignore_loss = _get_row_value(batch, "ignore_loss", idx)
    vntl_result = _extract_vntl_pair(raw_text, raw_ignore_loss)
    if vntl_result is not None:
        metadata, ja_text, en_text = vntl_result
        return _render_vntl_chat(metadata, ja_text, en_text)

    # Plain text fallback
    if _is_non_empty_text(raw_text):
        return raw_text.removeprefix('<bos>')

    # Key-pair fallback
    candidate_pairs = [
        ("prompt", "response"), ("instruction", "output"),
        ("question", "answer"), ("input", "output"), ("src", "tgt"),
    ]
    for user_key, assistant_key in candidate_pairs:
        user_text = _get_row_value(batch, user_key, idx)
        assistant_text = _get_row_value(batch, assistant_key, idx)
        if _is_non_empty_text(user_text) and _is_non_empty_text(assistant_text):
            return _render_chat_from_pair(user_text, assistant_text)

    return ""


def _is_usable_row(row):
    convos = row.get("conversations", None)
    if isinstance(convos, list) and len(convos) > 0:
        return True
    text = row.get("text", None)
    if _is_non_empty_text(text):
        return True
    candidate_pairs = [
        ("prompt", "response"), ("instruction", "output"),
        ("question", "answer"), ("input", "output"), ("src", "tgt"),
    ]
    for user_key, assistant_key in candidate_pairs:
        if _is_non_empty_text(row.get(user_key, None)) and _is_non_empty_text(row.get(assistant_key, None)):
            return True
    return False


def formatting_prompts_func(examples):
    n_rows = 0
    for values in examples.values():
        if isinstance(values, list):
            n_rows = len(values)
            break
    texts = [_row_to_text_from_batch(examples, i) for i in range(n_rows)]
    return {"text": texts}


## Filter & Build Datasets


In [7]:
from collections import OrderedDict

# Keep source datasets stable so this section is safe to rerun.
stage1_source = shisa_stage1_raw
stage2_source = stage2_raw

eval_source_pools = OrderedDict([
    ("retention_shisa", RAW_EVAL_POOLS["retention_shisa"]),
    ("vntl_q_dev", RAW_EVAL_POOLS["vntl_q_dev"]),
    ("vntl_q_stress", RAW_EVAL_POOLS["vntl_q_stress"]),
    ("vntl_harubench", RAW_EVAL_POOLS["vntl_harubench"]),
])

stage1_filtered = stage1_source.filter(_is_usable_row)
stage2_filtered = stage2_source.filter(_is_usable_row)
eval_filtered = OrderedDict(
    (track_name, ds.filter(_is_usable_row))
    for track_name, ds in eval_source_pools.items()
)

# Stratified eval subsets
if len(eval_filtered["retention_shisa"]) > SHISA_HOLDOUT_ROWS:
    eval_filtered["retention_shisa"] = stratified_select(eval_filtered["retention_shisa"], SHISA_HOLDOUT_ROWS, RANDOM_SEED, shisa_stratify_key)
if len(eval_filtered["vntl_q_dev"]) > VNTL_Q_DEV_ROWS:
    eval_filtered["vntl_q_dev"] = stratified_select(eval_filtered["vntl_q_dev"], VNTL_Q_DEV_ROWS, RANDOM_SEED, vntl_stratify_key)
if len(eval_filtered["vntl_harubench"]) > HARUBENCH_EVAL_ROWS:
    eval_filtered["vntl_harubench"] = stratified_select(eval_filtered["vntl_harubench"], HARUBENCH_EVAL_ROWS, RANDOM_SEED, vntl_stratify_key)

# NSFW filtering
if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_STAGE2:
    stage2_before = len(stage2_filtered)
    stage2_filtered = stage2_filtered.filter(lambda row: not _is_nsfw_row_tuned(row, NSFW_FILTER_LEVEL_STAGE2), load_from_cache_file=False)
    print(f"NSFW filter (stage2, level={NSFW_FILTER_LEVEL_STAGE2}): removed {stage2_before - len(stage2_filtered)} rows")

if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_EVAL_TRACKS:
    for track_name in list(eval_filtered.keys()):
        before = len(eval_filtered[track_name])
        eval_filtered[track_name] = eval_filtered[track_name].filter(lambda row: not _is_nsfw_row_tuned(row, NSFW_FILTER_LEVEL_EVAL_TRACKS), load_from_cache_file=False)
        print(f"NSFW filter ({track_name}, level={NSFW_FILTER_LEVEL_EVAL_TRACKS}): removed {before - len(eval_filtered[track_name])} rows")

# Format text datasets
def _format_text_dataset(ds):
    formatted = ds.map(formatting_prompts_func, batched=True, keep_in_memory=True)
    return formatted.filter(lambda x: _is_non_empty_text(x["text"]))

dataset = _format_text_dataset(stage1_filtered)
stage2_dataset = _format_text_dataset(stage2_filtered)
formatted_eval_sources = OrderedDict(
    (name, _format_text_dataset(ds)) for name, ds in eval_filtered.items()
)

# Post-format NSFW pass on formatted text
if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_STAGE2:
    stage2_before_post = len(stage2_dataset)
    stage2_dataset = stage2_dataset.filter(lambda row: not _is_nsfw_text_tuned(row.get("text", ""), level=NSFW_FILTER_LEVEL_STAGE2), load_from_cache_file=False)
    print(f"Post-format NSFW filter (stage2): removed {stage2_before_post - len(stage2_dataset)} rows")

if ENABLE_NSFW_FILTER and APPLY_NSFW_FILTER_TO_EVAL_TRACKS:
    for track_name in list(formatted_eval_sources.keys()):
        before = len(formatted_eval_sources[track_name])
        formatted_eval_sources[track_name] = formatted_eval_sources[track_name].filter(lambda row: not _is_nsfw_text_tuned(row.get("text", ""), level=NSFW_FILTER_LEVEL_EVAL_TRACKS), load_from_cache_file=False)
        print(f"Post-format NSFW filter ({track_name}): removed {before - len(formatted_eval_sources[track_name])} rows")

EVAL_TRACK_SOURCES = formatted_eval_sources
EVAL_TRACKS = build_eval_tracks(EVAL_TRACK_SOURCES)

print(f"\nFormatted Stage1 rows: {len(dataset)} | Stage2 rows: {len(stage2_dataset)}")
print("\nEval tracks:")
for track_name, ds in EVAL_TRACKS.items():
    print(f"  {track_name}: {len(ds)} rows")


Filter: 12790 examples [00:00, 15809.43 examples/s]       


NSFW filter (stage2, level=explicit_only): removed 2 rows


Filter: 512 examples [00:00, 14483.02 examples/s]       


NSFW filter (retention_shisa, level=minimal): removed 0 rows


Filter: 512 examples [00:00, 14211.96 examples/s]       


...

Post-format NSFW filter (vntl_q_stress): removed 0 rows


Filter: 100%|██████████| 256/256 [00:00<00:00, 19966.93 examples/s]


Post-format NSFW filter (vntl_harubench): removed 0 rows


Filter: 100%|██████████| 256/256 [00:00<00:00, 20729.81 examples/s]


Formatted Stage1 rows: 12000 | Stage2 rows: 6393

Eval tracks:
  retention_shisa.general: 256 rows
  vntl_q_dev.general: 251 rows
  vntl_q_dev.explicit: 5 rows
  vntl_q_stress.general: 126 rows
  vntl_q_stress.explicit: 2 rows
  vntl_harubench.general: 246 rows
  vntl_harubench.explicit: 10 rows


## Dataset Diagnostics

Quick checks for schema quality, token-length distribution, and sample previews.


In [8]:
import numpy as np
from collections import Counter
from statistics import mean


def _token_length_stats(ds, text_field="text", sample_rows=2000):
    lengths = []
    n = min(len(ds), sample_rows)
    for i in range(n):
        text = ds[i].get(text_field, "")
        if isinstance(text, str) and text.strip():
            token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            lengths.append(len(token_ids))
    if not lengths:
        return {"count": 0}
    return {
        "count": len(lengths),
        "mean": float(np.mean(lengths)),
        "p50": float(np.percentile(lengths, 50)),
        "p90": float(np.percentile(lengths, 90)),
        "p95": float(np.percentile(lengths, 95)),
        "p99": float(np.percentile(lengths, 99)),
        "max": int(np.max(lengths)),
    }


for name, ds in [
    ("stage1_formatted", dataset),
    ("stage2_formatted", stage2_dataset),
] + [(f"{tn}_formatted", d) for tn, d in EVAL_TRACK_SOURCES.items()]:
    stats = _token_length_stats(ds, sample_rows=1500)
    over_max = sum(1 for v in [stats.get("max", 0)] if v > MAX_SEQ_LENGTH)
    print(f"\n{name}: count={stats['count']} mean={stats.get('mean', 0):.0f} p90={stats.get('p90', 0):.0f} p99={stats.get('p99', 0):.0f} max={stats.get('max', 0)}")
    if stats.get("p99", 0) > MAX_SEQ_LENGTH:
        print(f"  WARNING: p99 ({stats['p99']:.0f}) exceeds MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH})")



stage1_formatted: count=1500 mean=424 p90=863 p99=2730 max=3663

stage2_formatted: count=1500 mean=473 p90=626 p99=1377 max=3277

retention_shisa_formatted: count=256 mean=449 p90=1028 p99=2441 max=3756

vntl_q_dev_formatted: count=256 mean=530 p90=587 p99=624 max=631

vntl_q_stress_formatted: count=128 mean=530 p90=591 p99=644 max=675

vntl_harubench_formatted: count=256 mean=531 p90=597 p99=632 max=668


In [ ]:
# Sample previews
print("=== Stage1 sample ===\n")
print(dataset[min(3, len(dataset) - 1)]["text"][:1200])
print("\n=== Stage2 (VNTL) sample ===\n")
print(stage2_dataset[0]["text"][:1200])

# NSFW audit
def _audit_residual_nsfw(ds, level, name, max_preview=3):
    flagged = [i for i in range(len(ds)) if _is_nsfw_text_tuned(ds[i].get("text", ""), level=level)]
    print(f"\n{name}: flagged {len(flagged)} / {len(ds)}")
    for idx in flagged[:max_preview]:
        snippet = ds[idx].get("text", "")[:260].replace("\n", " ")
        print(f"  idx={idx} snippet={snippet}")

_audit_residual_nsfw(stage2_dataset, NSFW_FILTER_LEVEL_STAGE2, "stage2_dataset")
for track_name, ds in EVAL_TRACK_SOURCES.items():
    _audit_residual_nsfw(ds, NSFW_FILTER_LEVEL_EVAL_TRACKS, track_name)


## Model Setup

Load Gemma 4 in 4-bit and keep LoRA detached for pre-train benchmarking.


In [9]:
import gc

if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

visible_gpus = torch.cuda.device_count()
print(f"Visible GPUs: {visible_gpus}")
for i in range(visible_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {props.name} | VRAM {props.total_memory / 1024**3:.2f} GiB")

extra_load_kwargs = {}
if USE_MULTI_GPU_MODEL_SHARDING and visible_gpus > 1:
    max_memory = {}
    for i in range(visible_gpus):
        total_gib = int(torch.cuda.get_device_properties(i).total_memory / 1024**3)
        reserve = GPU0_RESERVED_GIB if i == 0 else OTHER_GPU_RESERVED_GIB
        alloc = max(8, total_gib - reserve)
        max_memory[i] = f"{alloc}GiB"
    max_memory["cpu"] = "128GiB"
    extra_load_kwargs = {"device_map": MULTI_GPU_DEVICE_MAP, "max_memory": max_memory}
    print(f"Model split: device_map={MULTI_GPU_DEVICE_MAP} max_memory={max_memory}")
else:
    print("Using single-device model placement.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=LOAD_IN_4BIT, **extra_load_kwargs,
    )
except ValueError as e:
    if "max_memory" in extra_load_kwargs:
        print(f"max_memory rejected ({e}). Retrying with device_map only.")
        fallback_kwargs = {k: v for k, v in extra_load_kwargs.items() if k != "max_memory"}
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
            dtype=None, load_in_4bit=LOAD_IN_4BIT, **fallback_kwargs,
        )
    else:
        raise
except TypeError as e:
    print(f"Multi-GPU kwargs not accepted ({e}). Falling back to default load.")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=LOAD_IN_4BIT,
    )

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

BASE_BENCHMARKS = None
STAGE1_BENCHMARKS = None
STAGE2_BENCHMARKS = None
CHECKPOINT_BENCHMARKS = {}
CHECKPOINT_SUMMARIES = {}
LORA_ATTACHED = False

param_devices = sorted({str(p.device) for p in model.parameters()})
print(f"\nModel: {MODEL_NAME} | Devices: {param_devices}")
free_vram = torch.cuda.mem_get_info()[0] / 1e9
print(f"Free VRAM: {free_vram:.1f} GB")


Visible GPUs: 2
  cuda:0 NVIDIA GeForce RTX 5090 | VRAM 31.84 GiB
  cuda:1 NVIDIA GeForce RTX 3090 | VRAM 24.00 GiB
Model split: device_map=balanced max_memory={0: '26GiB', 1: '20GiB', 'cpu': '128GiB'}
==((====))==  Unsloth 2026.4.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 2. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 1188/1188 [00:21<00:00, 54.68it/s] 



Model: unsloth/gemma-4-31B-it | Devices: ['cuda:0', 'cuda:1']
Free VRAM: 23.8 GB


In [10]:
def ensure_lora_model():
    global model, tokenizer, LORA_ATTACHED

    model_missing = ("model" not in globals()) or (model is None)
    if model_missing:
        print("Base model missing. Reloading...")
        reload_kwargs = {}
        visible_gpus = torch.cuda.device_count()
        if USE_MULTI_GPU_MODEL_SHARDING and visible_gpus > 1:
            reload_kwargs["device_map"] = MULTI_GPU_DEVICE_MAP
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
            dtype=None, load_in_4bit=LOAD_IN_4BIT, **reload_kwargs,
        )
        tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
        LORA_ATTACHED = False
        print("Base model reloaded.")

    if LORA_ATTACHED:
        print("LoRA adapters already attached.")
        return

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    LORA_ATTACHED = True
    print(f"LoRA attached: r={LORA_R} alpha={LORA_ALPHA}")

print("LoRA helper ready.")


LoRA helper ready.


## Benchmark Helpers

Evaluate baseline and post-training quality with held-out benchmark rows.
Includes both loss/perplexity metrics and generation-based targeted checks.


In [ ]:
import math
import statistics
import time


def _mask_non_response_tokens(input_ids, labels, tokenizer_obj, texts, response_marker="<|turn>model\n", end_marker="<turn|>"):
    """Mask all tokens outside assistant responses using string-level offset mapping."""
    for b in range(labels.shape[0]):
        text = texts[b] if b < len(texts) else ""
        # Find all response ranges in the raw text string
        resp_char_ranges = []
        search_start = 0
        while True:
            marker_pos = text.find(response_marker, search_start)
            if marker_pos == -1:
                break
            resp_start_char = marker_pos + len(response_marker)
            end_pos = text.find(end_marker, resp_start_char)
            if end_pos == -1:
                resp_end_char = len(text)
            else:
                resp_end_char = end_pos
            resp_char_ranges.append((resp_start_char, resp_end_char))
            search_start = resp_end_char + len(end_marker) if end_pos != -1 else len(text)

        if not resp_char_ranges:
            continue  # No response markers found, don't mask anything (score all tokens)

        # Map character offsets to token positions using the tokenizer
        # Tokenize the full text and get offset mapping
        inner_tok = getattr(tokenizer_obj, "tokenizer", tokenizer_obj)
        enc_with_offsets = inner_tok(text, return_offsets_mapping=True, add_special_tokens=True)
        offsets = enc_with_offsets["offset_mapping"]

        # Build a mask: True = should be masked (set to -100)
        seq_len = labels.shape[1]
        mask = [True] * seq_len

        for tok_idx in range(min(len(offsets), seq_len)):
            tok_start, tok_end = offsets[tok_idx]
            if tok_start == tok_end == 0:
                continue  # special token, mask it
            # Check if this token overlaps with any response range
            for resp_start, resp_end in resp_char_ranges:
                if tok_start >= resp_start and tok_end <= resp_end:
                    mask[tok_idx] = False
                    break

        import torch as _torch
        mask_tensor = _torch.tensor(mask, dtype=_torch.bool, device=labels.device)[:seq_len]
        labels[b][mask_tensor] = -100

    return labels


def compute_eval_loss_direct(model_obj, tokenizer_obj, eval_ds, batch_size=1, max_length=MAX_SEQ_LENGTH):
    texts = [x["text"] for x in eval_ds if isinstance(x.get("text", None), str) and x["text"].strip()]
    if len(texts) == 0:
        return float("nan")

    total_neg_log_likelihood = 0.0
    total_tokens = 0
    model_obj.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer_obj(
            text=batch_texts, return_tensors="pt",
            padding=True, truncation=True, max_length=max_length,
        )
        enc = {k: v.to("cuda") for k, v in enc.items()}

        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        labels = _mask_non_response_tokens(enc["input_ids"], labels, tokenizer_obj, batch_texts)
        valid_tokens = int((labels != -100).sum().item())
        if valid_tokens == 0:
            continue

        with torch.inference_mode():
            outputs = model_obj(**enc, labels=labels)
            loss_value = float(outputs.loss.item())

        total_neg_log_likelihood += loss_value * valid_tokens
        total_tokens += valid_tokens

    if total_tokens == 0:
        return float("nan")
    return total_neg_log_likelihood / total_tokens


BENCHMARK_PROMPTS = [
    "Translate this natural VN line to smooth English while preserving tone: \u300c\u305d\u3093\u306a\u3053\u3068\u8a00\u308f\u308c\u3066\u3082\u2026\u2026\u56f0\u308b\u3088\u300d",
    "Rewrite this EN line with softer VN tone: I guess I can stay for a bit longer.",
    "Continue this dialogue naturally in VN style: [A] Did you really come all this way? [B]",
]


def run_generation_benchmark(model_obj, tokenizer_obj, prompts, max_new_tokens=80):
    def measure_generation(prompt):
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        inputs = getattr(tokenizer_obj, "tokenizer", tokenizer_obj).apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to("cuda")

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            outputs = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        generated_tokens = int(outputs.shape[-1] - inputs["input_ids"].shape[-1])
        tok_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
        return elapsed, generated_tokens, tok_per_sec

    _ = measure_generation("Warmup run for generation benchmark.")
    results = [measure_generation(p) for p in prompts]
    tps = [r[2] for r in results]
    return {
        "avg_latency_sec": statistics.mean([r[0] for r in results]),
        "avg_generated_tokens": statistics.mean([r[1] for r in results]),
        "avg_tokens_per_sec": statistics.mean(tps),
        "median_tokens_per_sec": statistics.median(tps),
    }


def generate_and_evaluate(model_obj, tokenizer_obj, eval_ds, max_samples=32):
    """Generate translations on VNTL eval samples and run targeted checks."""
    n = min(max_samples, len(eval_ds))
    results = []

    model_obj.eval()
    for i in range(n):
        raw_text = eval_ds[i].get("text", "")
        if "<<JAPANESE>>" in raw_text and "<<ENGLISH>>" in raw_text:
            pair = _extract_vntl_pair(raw_text, eval_ds[i].get("ignore_loss", None))
            if pair is None:
                continue
            metadata, ja_text, en_text = pair
        else:
            continue

        user_content = ""
        if metadata:
            user_content += f"<character_reference>\n{metadata}\n</character_reference>\n\n"
        user_content += f"Japanese script:\n{ja_text}"

        messages = [
            {"role": "system", "content": VN_TRAINING_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
        inputs = getattr(tokenizer_obj, "tokenizer", tokenizer_obj).apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to("cuda")

        with torch.inference_mode():
            outputs = model_obj.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True)

        pred_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        prediction_en = tokenizer_obj.decode(pred_ids, skip_special_tokens=True).strip()

        check = run_targeted_checks(ja_text, en_text, prediction_en)
        results.append(check)

    if not results:
        return {}

    summary = summarize_targeted_results(results)
    summary["samples_evaluated"] = len(results)
    return summary


def run_track_benchmark(model_obj, tokenizer_obj, eval_ds, track_name, checkpoint_name, generation_metrics):
    eval_loss = compute_eval_loss_direct(model_obj, tokenizer_obj, eval_ds)
    perplexity = math.exp(eval_loss) if isinstance(eval_loss, (float, int)) and math.isfinite(eval_loss) and eval_loss < 20 else float("inf")
    metrics = {
        "rows": len(eval_ds), "eval_loss": eval_loss,
        "perplexity": perplexity, **generation_metrics,
    }
    print(f"  [{checkpoint_name}] {track_name}: loss={eval_loss:.4f} ppl={perplexity:.4f}")
    return metrics


def run_checkpoint_benchmarks(model_obj, tokenizer_obj, eval_tracks, checkpoint_name, base_metrics=None, stage1_metrics=None):
    if not eval_tracks:
        raise RuntimeError("EVAL_TRACKS is empty.")

    generation_metrics = run_generation_benchmark(model_obj, tokenizer_obj, BENCHMARK_PROMPTS, max_new_tokens=80)
    print(f"\n[{checkpoint_name}] generation: {generation_metrics['avg_tokens_per_sec']:.2f} tok/s")

    metrics_by_track = {}
    for track_name, eval_ds in eval_tracks.items():
        metrics_by_track[track_name] = run_track_benchmark(
            model_obj, tokenizer_obj, eval_ds, track_name, checkpoint_name, generation_metrics,
        )

    summary = summarize_checkpoint_metrics(
        checkpoint_name, metrics_by_track,
        base_metrics=base_metrics, stage1_metrics=stage1_metrics,
        retention_track_prefix=RETENTION_TRACK_PREFIX,
        eval_loss_tolerance=RETENTION_EVAL_LOSS_TOLERANCE,
        perplexity_tolerance=RETENTION_PERPLEXITY_TOLERANCE,
    )
    print_checkpoint_summary(summary)

    for track_name, eval_ds_source in EVAL_TRACK_SOURCES.items():
        if "vntl" in track_name:
            targeted = generate_and_evaluate(model_obj, tokenizer_obj, eval_ds_source, max_samples=32)
            if targeted:
                summary.setdefault("targeted_checks", {})[track_name] = targeted
                print(f"  [{checkpoint_name}] {track_name} targeted: placeholder={targeted.get('placeholder_exact_rate', 0):.2f} honorific={targeted.get('honorific_alignment_rate', 0):.2f} short={targeted.get('suspiciously_short_rate', 0):.2f}")

    return metrics_by_track, summary


## Base Checkpoint Benchmark

Run before LoRA attachment to capture true base performance.

In [12]:
if len(EVAL_TRACKS) == 0:
    raise RuntimeError("EVAL_TRACKS is empty. Re-run dataset build cells before benchmarking.")

if LORA_ATTACHED:
    raise RuntimeError("LoRA is already attached. Restart and run in order to capture true base benchmark.")

BASE_BENCHMARKS, base_summary = run_checkpoint_benchmarks(model, tokenizer, EVAL_TRACKS, checkpoint_name="base")
CHECKPOINT_BENCHMARKS["base"] = BASE_BENCHMARKS
CHECKPOINT_SUMMARIES["base"] = base_summary



[base] generation: 5.67 tok/s
  [base] retention_shisa.general: loss=2.8437 ppl=17.1788
  [base] vntl_q_dev.general: loss=6.6291 ppl=756.7994
  [base] vntl_q_dev.explicit: loss=6.7059 ppl=817.1760
  [base] vntl_q_stress.general: loss=6.7208 ppl=829.4608
  [base] vntl_q_stress.explicit: loss=6.4600 ppl=639.0496
  [base] vntl_harubench.general: loss=6.6402 ppl=765.2149
  [base] vntl_harubench.explicit: loss=7.1843 ppl=1318.5755

Checkpoint summary: base
- retention_shisa.general
  eval_loss=2.8437
  perplexity=17.1788
  avg_tokens_per_sec=5.67
- vntl_q_dev.general
  eval_loss=6.6291
  perplexity=756.7994
  avg_tokens_per_sec=5.67
- vntl_q_dev.explicit
  eval_loss=6.7059
  perplexity=817.1760
  avg_tokens_per_sec=5.67
- vntl_q_stress.general
  eval_loss=6.7208
  perplexity=829.4608
  avg_tokens_per_sec=5.67
- vntl_q_stress.explicit
  eval_loss=6.4600
  perplexity=639.0496
  avg_tokens_per_sec=5.67
- vntl_harubench.general
  eval_loss=6.6402
  perplexity=765.2149
  avg_tokens_per_sec=5.67

## Two-Stage Training

Stage 1: Shisa warmup. Stage 2: VNTL specialization + Shisa replay.


In [13]:
import os, sys
import torch.nn.functional as F


def _install_tensorwrapper_fallback(chunk_tokens=1024):
    def _safe_unsloth_fused_ce_loss(
        trainer, hidden_states, lm_head_weight, lm_head_bias, labels,
        mask=None, n_items=None, scaling=None, target_gb=None,
        torch_compile=True, overwrite=False, **kwargs,
    ):
        device = lm_head_weight.device
        hidden_states = hidden_states.to(device=device, dtype=lm_head_weight.dtype)

        shifted_states = hidden_states[..., :-1, :].contiguous()
        shifted_labels = labels[..., 1:].contiguous().clone()
        if mask is not None:
            shifted_mask = mask[..., 1:].to(device=shifted_labels.device)
            shifted_labels = shifted_labels.masked_fill(shifted_mask == 0, -100)

        flat_states = shifted_states.view(-1, shifted_states.shape[-1])
        flat_labels = shifted_labels.view(-1).to(device=device)

        logit_scale_multiply = kwargs.get("logit_scale_multiply", 0)
        logit_scale_divide = kwargs.get("logit_scale_divide", 0)
        logit_softcapping = kwargs.get("logit_softcapping", 0)

        total_loss = torch.zeros((), device=device, dtype=torch.float32)
        total_valid = torch.zeros((), device=device, dtype=torch.float32)

        for start in range(0, flat_labels.numel(), chunk_tokens):
            end = min(start + chunk_tokens, flat_labels.numel())
            hs_chunk = flat_states[start:end]
            lb_chunk = flat_labels[start:end]

            logits = F.linear(hs_chunk, lm_head_weight, lm_head_bias).float()
            if logit_scale_multiply not in (None, 0):
                logits = logits * logit_scale_multiply
            if logit_scale_divide not in (None, 0):
                logits = logits / logit_scale_divide
            if logit_softcapping not in (None, 0):
                logits = logits / logit_softcapping
                logits = torch.tanh(logits)
                logits = logits * logit_softcapping

            chunk_loss = F.cross_entropy(logits, lb_chunk, ignore_index=-100, reduction="sum")
            total_loss = total_loss + chunk_loss
            total_valid = total_valid + (lb_chunk != -100).sum().to(torch.float32)

        if n_items is not None:
            denom = n_items.to(device=device, dtype=torch.float32) if torch.is_tensor(n_items) else torch.tensor(float(n_items), device=device, dtype=torch.float32)
        else:
            denom = torch.clamp(total_valid, min=1.0)

        return total_loss / denom

    patched = 0
    for mod_name, mod in list(sys.modules.items()):
        if mod is None:
            continue
        if mod_name.startswith("unsloth_compiled_module_") and hasattr(mod, "unsloth_fused_ce_loss"):
            setattr(mod, "unsloth_fused_ce_loss", _safe_unsloth_fused_ce_loss)
            patched += 1

    try:
        import unsloth_zoo.loss_utils as _loss_utils
        _loss_utils.unsloth_fused_ce_loss = _safe_unsloth_fused_ce_loss
    except Exception:
        pass

    print(f"TensorWrapper fallback CE installed in {patched} compiled module(s); chunk_tokens={chunk_tokens}")


_fallback_chunk_tokens = int(os.environ.get("UNSLOTH_SAFE_CE_CHUNK_TOKENS", "1024"))
_install_tensorwrapper_fallback(chunk_tokens=max(128, _fallback_chunk_tokens))

if len(dataset) == 0:
    raise RuntimeError("Stage 1 dataset is empty.")
if len(stage2_dataset) == 0:
    raise RuntimeError("Stage 2 dataset is empty.")

# ── Check for existing Stage 1 checkpoint ────────────────────────────────────
stage1_checkpoint_exists = Path(OUTPUT_DIR).joinpath("adapter_config.json").exists()
if stage1_checkpoint_exists:
    print(f"Stage 1 checkpoint found at {OUTPUT_DIR}. Skipping Stage 1 training.")
    print("Delete the checkpoint directory to retrain Stage 1.")
else:
    ensure_lora_model()

    trainer_stage1 = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=dataset, eval_dataset=None,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            warmup_steps=max(8, STAGE1_MAX_STEPS // 10),
            max_steps=STAGE1_MAX_STEPS,
            learning_rate=STAGE1_LR,
            logging_steps=10,
            save_steps=max(20, STAGE1_MAX_STEPS // 4),
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            max_grad_norm=0.3,
            seed=RANDOM_SEED,
            report_to="none",
            neftune_noise_alpha=5,
        ),
    )
    trainer_stage1 = train_on_responses_only(
        trainer_stage1,
        instruction_part="<|turn>user\n",
        response_part="<|turn>model\n",
    )

    print("Starting Stage 1 training...")
    trainer_stats_stage1 = trainer_stage1.train()
    trainer_stage1.save_model(OUTPUT_DIR)
    print(f"Stage 1 adapter saved to: {OUTPUT_DIR}")

STAGE1_BENCHMARKS, stage1_summary = run_checkpoint_benchmarks(
    model, tokenizer, EVAL_TRACKS, checkpoint_name="stage1_shisa",
    base_metrics=BASE_BENCHMARKS,
)
CHECKPOINT_BENCHMARKS["stage1_shisa"] = STAGE1_BENCHMARKS
CHECKPOINT_SUMMARIES["stage1_shisa"] = stage1_summary

# ── VRAM cleanup between stages ─────────────────────────────────────────────
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"Free VRAM after Stage 1: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

# ── Stage 2 ──────────────────────────────────────────────────────────────────
stage2_checkpoint_exists = Path(STAGE2_OUTPUT_DIR).joinpath("adapter_config.json").exists()
if stage2_checkpoint_exists:
    print(f"Stage 2 checkpoint found at {STAGE2_OUTPUT_DIR}. Skipping Stage 2 training.")
else:
    ensure_lora_model()

    trainer_stage2 = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=stage2_dataset, eval_dataset=None,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            warmup_steps=max(8, STAGE2_MAX_STEPS // 10),
            max_steps=STAGE2_MAX_STEPS,
            learning_rate=STAGE2_LR,
            logging_steps=10,
            save_steps=max(20, STAGE2_MAX_STEPS // 4),
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            max_grad_norm=0.3,
            seed=RANDOM_SEED,
            report_to="none",
            neftune_noise_alpha=5,
        ),
    )
    trainer_stage2 = train_on_responses_only(
        trainer_stage2,
        instruction_part="<|turn>user\n",
        response_part="<|turn>model\n",
    )

    print("Starting Stage 2 training...")
    trainer_stats = trainer_stage2.train()
    trainer_stage2.save_model(STAGE2_OUTPUT_DIR)
    print(f"Stage 2 adapter saved to: {STAGE2_OUTPUT_DIR}")

STAGE2_BENCHMARKS, stage2_summary = run_checkpoint_benchmarks(
    model, tokenizer, EVAL_TRACKS, checkpoint_name="stage2_vntl_replay",
    base_metrics=BASE_BENCHMARKS, stage1_metrics=STAGE1_BENCHMARKS,
)
CHECKPOINT_BENCHMARKS["stage2_vntl_replay"] = STAGE2_BENCHMARKS
CHECKPOINT_SUMMARIES["stage2_vntl_replay"] = stage2_summary

trainer = trainer_stage2 if not stage2_checkpoint_exists else None


TensorWrapper fallback CE installed in 2 compiled module(s); chunk_tokens=1024


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients
LoRA attached: r=16 alpha=32


Filter: 100%|██████████| 12000/12000 [00:04<00:00, 2596.50 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Unsloth: Removed 1 out of 12000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.
Starting Stage 1 training...


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 2,998 | Num Epochs = 2 | Total steps = 400
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 133,481,728 of 31,406,568,240 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.715980
20,0.824316
30,0.801360
40,0.738148
50,0.751333
60,0.786676
70,0.657502
80,0.726288
90,0.728302
100,0.666603


c:\Users\li\miniconda3\envs\unsloth\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error _ssl.c:993: The handshake operation timed out - silently ignoring the lookup for the file config.json in unsloth/gemma-4-31B-it.
  warnings.warn(
c:\Users\li\miniconda3\envs\unsloth\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in unsloth/gemma-4-31B-it - will assume that the vocabulary was not modified.
  warnings.warn(


Stage 2 adapter saved to: outputs/stage2

[stage2_vntl_replay] generation: 3.58 tok/s
  [stage2_vntl_replay] retention_shisa.general: loss=0.7842 ppl=2.1906
  [stage2_vntl_replay] vntl_q_dev.general: loss=2.9724 ppl=19.5395
  [stage2_vntl_replay] vntl_q_dev.explicit: loss=3.0554 ppl=21.2304
  [stage2_vntl_replay] vntl_q_stress.general: loss=2.9884 ppl=19.8538
  [stage2_vntl_replay] vntl_q_stress.explicit: loss=2.9912 ppl=19.9105
  [stage2_vntl_replay] vntl_harubench.general: loss=2.9605 ppl=19.3078
  [stage2_vntl_replay] vntl_harubench.explicit: loss=3.1706 ppl=23.8215

Checkpoint summary: stage2_vntl_replay
- retention_shisa.general
  eval_loss=0.7842
  perplexity=2.1906
  avg_tokens_per_sec=3.58
- vntl_q_dev.general
  eval_loss=2.9724
  perplexity=19.5395
  avg_tokens_per_sec=3.58
- vntl_q_dev.explicit
  eval_loss=3.0554
  perplexity=21.2304
  avg_tokens_per_sec=3.58
- vntl_q_stress.general
  eval_loss=2.9884
  perplexity=19.8538
  avg_tokens_per_sec=3.58
- vntl_q_stress.explicit
  e

## Checkpoint Comparison Summary

In [14]:
def _print_delta_map(label, delta_map):
    if not delta_map:
        print(f"{label}: none")
        return
    print(label)
    for track_name, metrics in delta_map.items():
        print(f"  {track_name}")
        for metric_name, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"    {metric_name}: {value:.4f}")
            else:
                print(f"    {metric_name}: {value}")


for checkpoint_name in CHECKPOINT_ORDER:
    summary = CHECKPOINT_SUMMARIES.get(checkpoint_name)
    if summary is None:
        print(f"Missing summary for checkpoint: {checkpoint_name}")
        continue

    print_checkpoint_summary(summary)

    # Print targeted checks if available
    targeted = summary.get("targeted_checks", {})
    if targeted:
        print(f"\n  Targeted checks for {checkpoint_name}:")
        for track, checks in targeted.items():
            print(f"    {track}: placeholder={checks.get('placeholder_exact_rate', 0):.2f} "
                  f"speaker_tag={checks.get('speaker_tag_subset_rate', 0):.2f} "
                  f"honorific={checks.get('honorific_alignment_rate', 0):.2f} "
                  f"short={checks.get('suspiciously_short_rate', 0):.2f}")

    if checkpoint_name != "base":
        _print_delta_map(f"\n{checkpoint_name} delta_vs_base", summary.get("delta_vs_base"))
    if checkpoint_name == "stage2_vntl_replay":
        _print_delta_map(f"\n{checkpoint_name} delta_vs_stage1", summary.get("delta_vs_stage1"))
        flags = summary.get("retention_regression_flags", []) or []
        print("\nretention_regression_flags")
        if flags:
            for flag in flags:
                print(f"  - {flag}")
        else:
            print("  - none")



Checkpoint summary: base
- retention_shisa.general
  eval_loss=2.8437
  perplexity=17.1788
  avg_tokens_per_sec=5.67
- vntl_q_dev.general
  eval_loss=6.6291
  perplexity=756.7994
  avg_tokens_per_sec=5.67
- vntl_q_dev.explicit
  eval_loss=6.7059
  perplexity=817.1760
  avg_tokens_per_sec=5.67
- vntl_q_stress.general
  eval_loss=6.7208
  perplexity=829.4608
  avg_tokens_per_sec=5.67
- vntl_q_stress.explicit
  eval_loss=6.4600
  perplexity=639.0496
  avg_tokens_per_sec=5.67
- vntl_harubench.general
  eval_loss=6.6402
  perplexity=765.2149
  avg_tokens_per_sec=5.67
- vntl_harubench.explicit
  eval_loss=7.1843
  perplexity=1318.5755
  avg_tokens_per_sec=5.67
Retention regression flags: none

Checkpoint summary: stage1_shisa
- retention_shisa.general
  eval_loss=0.7971
  perplexity=2.2191
  avg_tokens_per_sec=3.58
- vntl_q_dev.general
  eval_loss=3.0636
  perplexity=21.4042
  avg_tokens_per_sec=3.58
- vntl_q_dev.explicit
  eval_loss=3.1352
  perplexity=22.9927
  avg_tokens_per_sec=3.58
- v

## Save Adapter

In [15]:
import shutil
from pathlib import Path

# ── Select best checkpoint based on VNTL eval loss ───────────────────────────
best_checkpoint = None
best_vntl_loss = float("inf")

for ckpt_name in CHECKPOINT_ORDER:
    summary = CHECKPOINT_SUMMARIES.get(ckpt_name)
    if summary is None:
        continue
    tracks = summary.get("tracks", {})
    # Average eval_loss across VNTL tracks (skip retention/shisa tracks)
    vntl_losses = [
        m["eval_loss"] for name, m in tracks.items()
        if "vntl" in name and isinstance(m.get("eval_loss"), (int, float))
        and m["eval_loss"] == m["eval_loss"]  # NaN check
    ]
    if vntl_losses:
        avg_loss = sum(vntl_losses) / len(vntl_losses)
        print(f"{ckpt_name}: avg VNTL eval_loss = {avg_loss:.4f}")
        if avg_loss < best_vntl_loss:
            best_vntl_loss = avg_loss
            best_checkpoint = ckpt_name

print()
print(f"Best checkpoint: {best_checkpoint} (avg VNTL loss: {best_vntl_loss:.4f})")

# ── Save best adapter ────────────────────────────────────────────────────────
Path(FINAL_ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

# If best is stage2 (most likely), the current model is already stage2
# If best is stage1, copy from the stage1 checkpoint directory
if best_checkpoint == "stage1_shisa" and Path(OUTPUT_DIR).exists():
    print(f"Best model is Stage 1. Copying adapter from {OUTPUT_DIR}")
    for f in Path(OUTPUT_DIR).glob("*"):
        if f.is_file():
            shutil.copy2(f, Path(FINAL_ADAPTER_DIR) / f.name)
elif best_checkpoint == "base":
    print("WARNING: Base model scored best on VNTL. Training may have degraded performance.")
    print("Saving Stage 2 adapter anyway for inspection.")
    model.save_pretrained(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
else:
    model.save_pretrained(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"Saved best LoRA adapter ({best_checkpoint}) to: {FINAL_ADAPTER_DIR}")

# Optional GGUF export
EXPORT_GGUF = False
if EXPORT_GGUF:
    gguf_dir = f"outputs/gemma4_{MODEL_VARIANT.lower()}_gguf"
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="Q8_0")
    print(f"Saved GGUF to {gguf_dir}")


base: avg VNTL eval_loss = 6.7234
stage1_shisa: avg VNTL eval_loss = 3.1111
stage2_vntl_replay: avg VNTL eval_loss = 3.0231

Best checkpoint: stage2_vntl_replay (avg VNTL loss: 3.0231)
Saved best LoRA adapter (stage2_vntl_replay) to: outputs/gemma4_31b_lora_final


## Quick Inference Smoke Test

In [25]:
# Debug: check why response masking doesn't match VNTL tokens
sample_vntl = EVAL_TRACKS.get('vntl_q_dev.general', EVAL_TRACKS.get(list(k for k in EVAL_TRACKS if 'vntl' in k)[0]))
sample_text = [x['text'] for x in sample_vntl if isinstance(x.get('text', None), str) and x['text'].strip()][0]

# Tokenize the sample text the same way compute_eval_loss_direct does
enc = tokenizer(text=sample_text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH)
ids = enc['input_ids'][0].tolist()

# Tokenize the marker the same way _tok_encode does
marker_enc = tokenizer(text='<|turn>model', add_special_tokens=False)
marker_ids = marker_enc['input_ids']
if hasattr(marker_ids, 'tolist'):
    marker_ids = marker_ids.tolist()
if isinstance(marker_ids, list) and len(marker_ids) > 0 and isinstance(marker_ids[0], list):
    marker_ids = marker_ids[0]

end_enc = tokenizer(text='<|turn>', add_special_tokens=False)
end_ids = end_enc['input_ids']
if hasattr(end_ids, 'tolist'):
    end_ids = end_ids.tolist()
if isinstance(end_ids, list) and len(end_ids) > 0 and isinstance(end_ids[0], list):
    end_ids = end_ids[0]

print(f'Marker "<|turn>model" encodes to: {marker_ids}')
print(f'End marker "<|turn>" encodes to: {end_ids}')
print(f'Sample token count: {len(ids)}')
print(f'First 50 token IDs: {ids[:50]}')

# Decode token by token to find where turn markers are
inner_tok = getattr(tokenizer, 'tokenizer', tokenizer)
print('\nFirst 50 tokens decoded:')
for j, tid in enumerate(ids[:50]):
    print(f'  [{j}] id={tid} -> {repr(inner_tok.decode([tid]))}')

# Search for the marker in the token sequence
found = False
for j in range(len(ids) - len(marker_ids) + 1):
    if ids[j:j + len(marker_ids)] == marker_ids:
        print(f'\nFound marker at position {j}!')
        found = True
        break
if not found:
    print(f'\nMarker NOT found in token sequence!')
    # Try to find <|turn> as a single special token
    for j, tid in enumerate(ids):
        decoded = inner_tok.decode([tid])
        if '<|turn>' in decoded or 'turn' in decoded.lower():
            print(f'  Token with "turn" at [{j}] id={tid} -> {repr(decoded)}')


Marker "<|turn>model" encodes to: [105, 4368]
End marker "<|turn>" encodes to: [105]
Sample token count: 479
First 50 token IDs: [236820, 18857, 236779, 30449, 236813, 107, 236840, 18857, 236842, 7343, 236787, 83749, 7782, 158853, 568, 242103, 236743, 242066, 240195, 236768, 1109, 41526, 236787, 38561, 107, 236840, 18857, 236842, 7343, 236787, 20210, 548, 48116, 708, 2220, 568, 237594, 239028, 236743, 239669, 237844, 236768, 1109, 41526, 236787, 36566, 1109, 15352, 1780, 236787]

First 50 tokens decoded:
  [0] id=236820 -> '<'
  [1] id=18857 -> 'character'
  [2] id=236779 -> '_'
  [3] id=30449 -> 'reference'
  [4] id=236813 -> '>'
  [5] id=107 -> '\n'
  [6] id=236840 -> '['
  [7] id=18857 -> 'character'
  [8] id=236842 -> ']'
  [9] id=7343 -> ' Name'
  [10] id=236787 -> ':'
  [11] id=83749 -> ' Kis'
  [12] id=7782 -> 'aki'
  [13] id=158853 -> ' Reina'
  [14] id=568 -> ' ('
  [15] id=242103 -> '妃'
  [16] id=236743 -> ' '
  [17] id=242066 -> '玲'
  [18] id=240195 -> '奈'
  [19] id=236768 -

In [29]:
from transformers import TextStreamer

test_ja = "\u300c\u305d\u3093\u306a\u3053\u3068\u8a00\u308f\u308c\u3066\u3082\u2026\u2026\u56f0\u308b\u3088\u300d"
messages = [
    {"role": "system", "content": VN_TRAINING_SYSTEM_PROMPT},
    {"role": "user", "content": f"Japanese script:\n{test_ja}"},
]
inner_tok = getattr(tokenizer, "tokenizer", tokenizer)
prompt_text = inner_tok.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False,
)
inputs = tokenizer(text=prompt_text, return_tensors="pt").to("cuda")

print("Generating translation...\n")
_ = model.generate(
    **inputs, max_new_tokens=96,
    do_sample=False,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)


Generating translation...

" " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " 

KeyboardInterrupt: 